# Advanced data processing of large datasets using the `map` function.

This Jupyter Notebook was originally made for the "ESTEEM3 Workshop: Electron diffraction for solving engineering problems" at the Norwegian University of Science and Technology (NTNU) in Trondheim. It aimed to teach the participants how to write and run their own data processing on large datasets in HyperSpy, using the `map` function.

This notebook assumes some background knowledge in how to HyperSpy.

## Requirements

- HyperSpy 1.7.1
- pyXem 0.14.2

## Author

- 2022/06/22: Magnus Nord, TEM-Gemini, Department of Physics, NTNU. The Research Council of Norway is acknowledged for partially funding the creation of this Jupyter Notebook via the InCoMa project (Project number: 315475). Partial funding was also received from ESTEEM3 via the European Union's Horizon 2020 grant number 823717.

### Plotting library and loading HyperSpy

In [ ]:
%matplotlib qt

In [ ]:
import hyperspy.api as hs

### Loading data

The dataset is a scanning precession electron diffraction (SPED) dataset, with 256 x 256 probe positions, 256 x 256 detector pixels, with 16-bit depth. The material is a CPU sample prepared using Focused Ion Beam (FIB).

It is fairly large, 8.6 GB, so we use `lazy=True`.

In [ ]:
s = hs.load("data/cpu_sample_sped.zspy", lazy=True)

## Advanced processing with `map`

Looking at the dataset, there are many different crystal structures. To the top left, there is monocrystalline Si, with the darker contrast being nano- or polycrystalline. Between these, there are two different kinds of amorphous materials.

In [ ]:
s.plot(norm="symlog")

### Finding specific patterns

We want to find the regions with the amorphous material having the largest "inner" ring, as seen in the probe position `(169, 106)`.

There are many ways we could do this, here we'll use an experimental pattern with the `match_template` function from `scikit-image` (`skimage`) to compare with each diffraction pattern.

First, we need to make a "template" by averaging a couple of the amorphous pattern we're looking for.

In [ ]:
s.plot(norm="symlog")

There is a fairly thick layer in the middle, so we use the `inav` slicing.

In [ ]:
s_amorphous = s.inav[170:173, 104:108].mean()

This is still a lazy signal, so use `compute` to actually load it into memory.

In [ ]:
s_amorphous.compute()

This gives us a nicer pattern to use for the correlation.

In [ ]:
s_amorphous.plot(norm="symlog")

#### Masking out the direct beam

The direct beam (or bright field disk), is typically the most intense feature in diffraction patterns and will dominate the template matching. However, it does not contain much information about the structure of the material. Thus, we would like to mask it using a disk.

First, we need to estimate the radius and location of the direct beam.

In [ ]:
s_amorphous.plot(norm="symlog")

A radius of `11` pixels should be fine. However, we use a radius of `11.5`, to make the shape of the mask array an even number.

In addition, the centre position is in (128, 128), which is the middle of the detector.

To do this, we utilize another function in `scikit-image`: `morphology.disk`.

In [ ]:
from skimage.morphology import disk

Since we want to use this a mask, we want the datatype to be boolean.

In [ ]:
binary_disk = disk(11.5, dtype=bool)

This returns a NumPy array with size of (24, 24).

In [ ]:
binary_disk.shape

However, the shape of the `binary_disk` array is currently too small to be used as an actual mask. We need zero-pad it. To do this, use a `NumPy` function `pad`.

In [ ]:
import numpy as np

`pad` works by adding zeros around the array which is passed to the function. Since the direct beam is in the middle of the detector (128, 128), and the mask array is (24, 24), we need to add 116 pixels of padding to each direction.

In [ ]:
center_beam_mask = np.pad(binary_disk, pad_width=116)

Currently, the center part of the mask is `True`, while the rest is `False`. For the function we're going to write, we want the opposite. `np.invert` flips the `False` to `True`, and `True` to `False`.

In [ ]:
center_beam_mask_invert = np.invert(center_beam_mask)

Next, we visualize the mask by making a new signal. This done by simply passing the NumPy array to a `Signal2D`.

In [ ]:
s_mask = hs.signals.Signal2D(center_beam_mask_invert)

In [ ]:
s_mask.plot()

#### Writing your own function

Finally, we need to our custom function which will be applied to each probe position. First, we need to import the function we want to use: `match_template`

In [ ]:
from skimage.feature import match_template

This function works by passing two arrays to the function. The second array has to be smaller or equal to the size of the first array.

Since the function wants a NumPy array, we give it `.data`, which is a NumPy array. To test the function, we pass it `s_amorphous.data` twice. Both as the image, and the template.

In [ ]:
match_template(s_amorphous.data, s_amorphous.data)

As expected, this gave us a perfect match!

Now, lets try with the mask. The easiest way of doing this is using the `*` operator.

In [ ]:
match_template(s_amorphous.data * s_mask.data, s_amorphous.data * s_mask.data)

As expected, this also gave us a perfect match.

Last, lets test this on a couple of diffraction patterns across the sample: the Si `(0, 0)`, some nanocrystalline areas `(23, 35)` and `(233, 76)`, the other amorphous region `(152, 78)`, and the amorphous region we're looking for `(114, 206)`.

In [ ]:
match_template(s.inav[0, 0].data * s_mask.data, s_amorphous.data * s_mask.data) # Si

In [ ]:
match_template(s.inav[23, 35].data * s_mask.data, s_amorphous.data * s_mask.data) # Nanocrystalline 1

In [ ]:
match_template(s.inav[233, 76].data * s_mask.data, s_amorphous.data * s_mask.data) # Nanocrystalline 2

In [ ]:
match_template(s.inav[152, 78].data * s_mask.data, s_amorphous.data * s_mask.data) # Amorphous 1

In [ ]:
match_template(s.inav[114, 206].data * s_mask.data, s_amorphous.data * s_mask.data) # Amorphous 2

The crystalline regions gave a low match, around 0.3. The "wrong" amorphous region 0.97, and the correct amorphous 0.98.

So it can clearly differentiate between mono/nano-crystalline and amorphous, but the values from the two amorphous structures is very similar. But lets write a function, and see how it performs on the full dataset.

The function definition in Python starts with a `def` and ends with a `:`, with the name of the function, and the parameters (or argument) which we'll use in-between. The first parameter has to be a single diffraction pattern (i.e. a single navigation position). We also need to pass the template and the mask.

In [ ]:
def afunction(image, template, mask):
    temp0 = image * mask
    temp1 = template * mask
    temp2 = match_template(temp0, temp1)
    output = temp2.squeeze()
    return output

With the function defined, and our mask and template prepared, we can finally run this using `map`! However, this is a fairly computationally expensive processing task. So to make this run a bit faster (and use less memory), you can bin the data using the `rebin` function.

#### Binning the data

Here, the data will be binned in both the detector and probe dimensions. Reducing the size of the data to `6.25%` of the original size.

Binning `s_mask`, we need to tell it that the datatype should be `bool`, since the `rebin` function will sum the pixels.

In [ ]:
s_mask = s_mask.rebin(scale=(2, 2), dtype=bool)

`s_amorphous` does not require any special settings.

In [ ]:
s_amorphous = s_amorphous.rebin(scale=(2, 2))

`s` is a large dataset, so we want to be a bit clever with this rebinning. The data is acquired at 12-bit depth, which means the maximum value is 4096. Since we're binning by (2, 2, 2, 2), we know the maximum value each pixel can have after the binning is 65536. So we only need to increase the datatype to 16-bit unsigned integer.

In [ ]:
s = s.rebin(scale=(2, 2, 2, 2), rechunk=False, dtype=np.uint16)

#### Running the processing

In [ ]:
s_template_match = s.map(afunction, template=s_amorphous, mask=s_mask, inplace=False)

This gave us a `LazySignal` with the same navigation shape as `s`. Since we want to visualize this as an image, transpose the signal using `.T`.

In [ ]:
s_template_match = s_template_match.T

In [ ]:
print(s_template_match)

Then we can `compute` it! Note that this might take 10 minutes or longer, especially if you haven't binned the data! Note: depending on your available memory and number of CPU cores, these calculation can lead to your computer crashing due to running out of memory. 

In [ ]:
s_template_match.compute(scheduler="processes")

Plotting this, we see that the amorphous regions are much brighter compared to the crystalline ones. In addition, the "correct" amorphous regions are also brighter compared to the "wrong" amorphous regions (depending on your computer monitor).

In [ ]:
s_template_match.plot()

To make it more visible, use the `vmin` and `vmax` parameters.

In [ ]:
s_template_match.plot(vmin=0.97, vmax=1.0)